In [ ]:
using System;
using System.Collections.Generic;

public interface IInstallable
{
    void Install();
}

public abstract class Component
{
    public string Name { get; set; }
    public string Manufacturer { get; set; }
    public int Price { get; set; }

    protected Component(string name, string manufacturer, int price)
    {
        Name = name;
        Manufacturer = manufacturer;
        Price = price;
    }

    public virtual void GetInfo()
    {
        Console.WriteLine($"  Компонент: {Name} | Производитель: {Manufacturer} | Цена: {Price} руб.");
    }

    public override string ToString()
    {
        return $"{Name} ({Manufacturer}) — {Price} руб.";
    }

    public static bool operator ==(Component a, Component b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;
        return a.Name == b.Name && a.Manufacturer == b.Manufacturer;
    }

    public static bool operator !=(Component a, Component b)
    {
        return !(a == b);
    }

    public override bool Equals(object obj)
    {
        return obj is Component c && this == c;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(Name, Manufacturer);
    }
}

public class CPU : Component, IInstallable
{
    public int Cores { get; set; }
    public double FrequencyGHz { get; set; }

    public CPU(string name, string manufacturer, int price, int cores, double frequency)
        : base(name, manufacturer, price)
    {
        Cores = cores;
        FrequencyGHz = frequency;
    }

    public override void GetInfo()
    {
        base.GetInfo();
        Console.WriteLine($"    [CPU] Ядер: {Cores} | Частота: {FrequencyGHz} ГГц");
    }

    public void Install()
    {
        Console.WriteLine($"  -> CPU '{Name}' установлен в сокет.");
    }
}

public class RAM : Component, IInstallable
{
    public int SizeGB { get; set; }
    public int SpeedMHz { get; set; }

    public RAM(string name, string manufacturer, int price, int sizeGB, int speedMHz)
        : base(name, manufacturer, price)
    {
        SizeGB = sizeGB;
        SpeedMHz = speedMHz;
    }

    public override void GetInfo()
    {
        base.GetInfo();
        Console.WriteLine($"    [RAM] Объем: {SizeGB} ГБ | Частота: {SpeedMHz} МГц");
    }

    public void Install()
    {
        Console.WriteLine($"  -> RAM '{Name}' установлена в слот памяти.");
    }
}

public class GPU : Component, IInstallable
{
    public int MemoryGB { get; set; }
    public string Interface { get; set; }

    public GPU(string name, string manufacturer, int price, int memoryGB, string intf)
        : base(name, manufacturer, price)
    {
        MemoryGB = memoryGB;
        Interface = intf;
    }

    public override void GetInfo()
    {
        base.GetInfo();
        Console.WriteLine($"    [GPU] Память: {MemoryGB} ГБ | Интерфейс: {Interface}");
    }

    public void Install()
    {
        Console.WriteLine($"  -> GPU '{Name}' установлена в слот {Interface}.");
    }
}

public class HDD : Component, IInstallable
{
    public int CapacityGB { get; set; }
    public int SpeedRPM { get; set; }

    public HDD(string name, string manufacturer, int price, int capacityGB, int speedRPM)
        : base(name, manufacturer, price)
    {
        CapacityGB = capacityGB;
        SpeedRPM = speedRPM;
    }

    public override void GetInfo()
    {
        base.GetInfo();
        Console.WriteLine($"    [HDD] Объем: {CapacityGB} ГБ | Скорость: {SpeedRPM} об/мин");
    }

    public void Install()
    {
        Console.WriteLine($"  -> HDD '{Name}' установлен в отсек для дисков.");
    }
}

public class Computer
{
    public string Model { get; set; }
    private List<Component> _components = new List<Component>();

    public Computer(string model)
    {
        Model = model;
    }

    public void AddComponent<T>(T component) where T : Component, IInstallable
    {
        _components.Add(component);
        component.Install();
        Console.WriteLine($"  -> Компонент '{component.Name}' добавлен в компьютер '{Model}'.");
    }

    public void RemoveComponent<T>(T component) where T : Component
    {
        if (_components.Contains(component))
        {
            _components.Remove(component);
            Console.WriteLine($"  -> Компонент '{component.Name}' удален из компьютера '{Model}'.");
        }
        else
        {
            Console.WriteLine($"  -> Компонент '{component.Name}' не найден в компьютере '{Model}'.");
        }
    }

    public int GetTotalPrice()
    {
        int sum = 0;
        foreach (var c in _components)
            sum += c.Price;
        return sum;
    }

    public void PrintAllComponents()
    {
        Console.WriteLine($"  Компьютер '{Model}' содержит {_components.Count} компонент(ов):");
        foreach (var c in _components)
            c.GetInfo();
    }

    public List<Component> GetComponents() => _components;
}

public static class ComputerFactory
{
    public static Computer CreateGamingPC()
    {
        var pc = new Computer("Gaming Beast");
        pc.AddComponent(new CPU("Intel Core i9-13900K", "Intel", 60000, 24, 3.0));
        pc.AddComponent(new RAM("Kingston Fury 32GB", "Kingston", 12000, 32, 3200));
        pc.AddComponent(new GPU("NVIDIA RTX 4090", "NVIDIA", 180000, 24, "PCIe 4.0"));
        pc.AddComponent(new HDD("Samsung 990 Pro 2TB", "Samsung", 20000, 2000, 7200));
        return pc;
    }

    public static Computer CreateOfficePC()
    {
        var pc = new Computer("Office Basic");
        pc.AddComponent(new CPU("Intel Core i3-12100", "Intel", 12000, 4, 3.3));
        pc.AddComponent(new RAM("Crucial 8GB", "Crucial", 3000, 8, 2666));
        pc.AddComponent(new HDD("WD Blue 1TB", "Western Digital", 4500, 1000, 7200));
        return pc;
    }

    public static Computer CreateCustomPC<T>(string model, T component)
        where T : Component, IInstallable
    {
        var pc = new Computer(model);
        pc.AddComponent(component);
        return pc;
    }

    public static Computer CreateFromComponents(string model, params Component[] components)
    {
        var pc = new Computer(model);
        foreach (var c in components)
        {
            if (c is IInstallable installable)
                installable.Install();
            pc.GetComponents().Add(c);
            Console.WriteLine($"  -> Компонент '{c.Name}' добавлен в компьютер '{model}'.");
        }
        return pc;
    }
}

try
{
    Console.WriteLine("\n--- 1. ПОЛИМОРФИЗМ (массив Component) ---");
    Component[] parts =
    {
        new CPU("AMD Ryzen 7 7800X3D", "AMD", 45000, 8, 4.2),
        new RAM("Corsair Vengeance 16GB", "Corsair", 8000, 16, 3600),
        new GPU("AMD Radeon RX 7900 XTX", "AMD", 110000, 24, "PCIe 4.0"),
        new HDD("Seagate Barracuda 4TB", "Seagate", 9000, 4000, 7200)
    };
    foreach (var p in parts)
        p.GetInfo();   


    Console.WriteLine("\n--- 2. ПЕРЕГРУЗКА ОПЕРАТОРОВ == и != ---");
    CPU cpu1 = new CPU("Intel Core i5", "Intel", 20000, 6, 2.9);
    CPU cpu2 = new CPU("Intel Core i5", "Intel", 25000, 8, 3.5);
    CPU cpu3 = new CPU("AMD Ryzen 5", "AMD", 18000, 6, 3.6);
    Console.WriteLine($"  cpu1 == cpu2 (одинаковые имя+произв.): {cpu1 == cpu2}");
    Console.WriteLine($"  cpu1 != cpu3: {cpu1 != cpu3}");
    Console.WriteLine($"  cpu1.Equals(cpu2): {cpu1.Equals(cpu2)}");

    Console.WriteLine("\n--- 3. ПЕРЕГРУЗКА ToString() ---");
    Console.WriteLine($"  {cpu1}");
    Console.WriteLine($"  {parts[2]}");

    Console.WriteLine("\n--- 4. COMPUTER: ДОБАВЛЕНИЕ / УДАЛЕНИЕ / СТОИМОСТЬ ---");
    var myPC = new Computer("Custom Build");
    myPC.AddComponent(cpu1);
    myPC.AddComponent(new RAM("Patriot 16GB", "Patriot", 7000, 16, 3000));
    myPC.AddComponent(new GPU("NVIDIA GTX 1660", "NVIDIA", 35000, 6, "PCIe 3.0"));
    myPC.PrintAllComponents();
    Console.WriteLine($"  Общая стоимость: {myPC.GetTotalPrice()} руб.");

    Console.WriteLine("\n  Удаляем GPU...");
    myPC.RemoveComponent(new GPU("NVIDIA GTX 1660", "NVIDIA", 35000, 6, "PCIe 3.0"));
    myPC.PrintAllComponents();
    Console.WriteLine($"  Общая стоимость после удаления: {myPC.GetTotalPrice()} руб.");

    Console.WriteLine("\n--- 5. COMPUTERFACTORY ---");
    var gaming = ComputerFactory.CreateGamingPC();
    gaming.PrintAllComponents();
    Console.WriteLine($"  Стоимость игрового ПК: {gaming.GetTotalPrice()} руб.");

    var office = ComputerFactory.CreateOfficePC();
    office.PrintAllComponents();
    Console.WriteLine($"  Стоимость офисного ПК: {office.GetTotalPrice()} руб.");

    var custom = ComputerFactory.CreateCustomPC("Mini PC", new CPU("Intel N100", "Intel", 5000, 4, 3.4));
    custom.PrintAllComponents();

    var multi = ComputerFactory.CreateFromComponents("Workstation",
        new CPU("AMD Threadripper", "AMD", 200000, 32, 4.0),
        new RAM("G.Skill 64GB", "G.Skill", 25000, 64, 3600),
        new HDD("NVMe 4TB", "WD", 30000, 4000, 0));
    multi.PrintAllComponents();
    Console.WriteLine($"  Стоимость рабочей станции: {multi.GetTotalPrice()} руб.");

    Console.WriteLine("\n==========================================================");
    Console.WriteLine("ДЕМОНСТРАЦИЯ ЗАВЕРШЕНА");
    Console.WriteLine("==========================================================");
}
catch (Exception ex)
{
    Console.WriteLine($"Ошибка выполнения -> {ex.Message}");
}


--- 1. ПОЛИМОРФИЗМ (массив Component) ---
  Компонент: AMD Ryzen 7 7800X3D | Производитель: AMD | Цена: 45000 руб.
    [CPU] Ядер: 8 | Частота: 4,2 ГГц
  Компонент: Corsair Vengeance 16GB | Производитель: Corsair | Цена: 8000 руб.
    [RAM] Объем: 16 ГБ | Частота: 3600 МГц
  Компонент: AMD Radeon RX 7900 XTX | Производитель: AMD | Цена: 110000 руб.
    [GPU] Память: 24 ГБ | Интерфейс: PCIe 4.0
  Компонент: Seagate Barracuda 4TB | Производитель: Seagate | Цена: 9000 руб.
    [HDD] Объем: 4000 ГБ | Скорость: 7200 об/мин

--- 2. ПЕРЕГРУЗКА ОПЕРАТОРОВ == и != ---
  cpu1 == cpu2 (одинаковые имя+произв.): True
  cpu1 != cpu3: True
  cpu1.Equals(cpu2): True

--- 3. ПЕРЕГРУЗКА ToString() ---
  Intel Core i5 (Intel) — 20000 руб.
  AMD Radeon RX 7900 XTX (AMD) — 110000 руб.

--- 4. COMPUTER: ДОБАВЛЕНИЕ / УДАЛЕНИЕ / СТОИМОСТЬ ---
  -> CPU 'Intel Core i5' установлен в сокет.
  -> Компонент 'Intel Core i5' добавлен в компьютер 'Custom Build'.
  -> RAM 'Patriot 16GB' установлена в слот памяти.
  ->